# Operaciones de aprendizaje automático
## Maestría en Inteligencia Artificial Aplicada

### TC5061 | Proyecto MiBici: propuesta y base reproducible | Colaborativo

#### **Equipo:** 7
#### **Nombre:** Fernando Arreola Treviño
#### **Matrícula:** A01840331

## Objetivo
Notebook de **entrenamiento de modelo** del proyecto MiBici.

La pregunta que me corresponde es la **P2 (clasificación)**: ¿Una estación estará en riesgo de quedarse sin bicicletas o sin anclajes disponibles en un periodo futuro?

La lógica que se reutilice se trasladará al módulo del proyecto; en este notebook se desarrolla y se documenta la evidencia que la justifica.

## Dataset
**Datos abiertos de MiBici**, el sistema de bicicleta pública del Área Metropolitana de Guadalajara. El portal publica:
- **Viajes por mes** (`datos_abiertos_AAAA_MM.csv`) desde diciembre de 2014. Cada registro es un viaje con identificador, datos del usuario, fecha y hora de inicio y fin, y estaciones de origen y destino.
- **Nomenclatura de estaciones** (`nomenclatura_AAAA_MM.csv`) con identificador, nombre, latitud, longitud y capacidad de cada estación.

- Fuente: [Datos abiertos de MiBici](https://mibici.net/es/datos-abiertos/)
- Rango analizado: **enero de 2020 a septiembre de 2026**, acordado por el equipo para homologar el EDA. Puede recortarse después con base en los hallazgos.

Se cuenta con lógica implementada para descargar los archivos en un rango especificado y en formato parquet, homologando la codificación y estandarización de fechas (`dataset.py`). Se usó este código para descargar los datos del 2020 al 2026 en la carpeta data/raw.

In [14]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import sys
import os
import glob
import pyarrow
from datetime import date
from pathlib import Path
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (average_precision_score, classification_report, confusion_matrix, log_loss,
                             precision_recall_curve, precision_recall_fscore_support)
from sklearn.utils.validation import check_is_fitted


print("Python      :", sys.version.split()[0])
print("pandas      :", pd.__version__)
print("numpy       :", np.__version__)
print("pyarrow     :", pyarrow.__version__)
print("matplotlib  :", matplotlib.__version__)

Python      : 3.11.5
pandas      : 3.0.5
numpy       : 1.26.4
pyarrow     : 25.0.1
matplotlib  : 3.11.2


## Configuración global


In [ ]:
from tc5061_proyecto_equipo07 import config as cfg

# Rutas del proyecto (el notebook se ejecuta desde notebooks/)
RUTA_PROYECTO = Path.cwd().parent
DATA_DIR = RUTA_PROYECTO / "data" / "processed"
ESTACIONES_DIR = RUTA_PROYECTO / "data" / "raw" / "estaciones"

# Horas sin servicio de MiBici
SIN_SERVICIO = [1, 2, 3, 4]

SEMILLA = 42

HORIZONTE_MIN =120
BIN_MIN = 30
H_BINS = HORIZONTE_MIN // BIN_MIN
ALFA = 0.5

FEATURES = ([f"neto_{k}_relativo" for k in (1, 2, 4, 8)] + [f"{c}_{k}" for c in ("salidas", "entradas") for k in (2, 4)] +
            ["cumsum_dia_relativo", "dpcapacity", "latitude", "longitude", "hora_sin", "hora_cos", "es_fin_semana"])
FEATURES_CON_HISTORICO = FEATURES + ["hist_sin_bicis", "hist_sin_anclajes"]

# Configuracion del modelo
# Partición común del proyecto (config.py): train hasta 2024-09, valid de 2024-10 a 2025-09 y test desde 2025-10.
# valid solo se usa para elegir los umbrales de decisión; test no se usa para decidir nada.
FRONTERA_VALID, FRONTERA_TEST, BRECHA_HORAS = cfg.FRONTERA_VALID, cfg.FRONTERA_TEST, cfg.BRECHA_HORAS
PESO_HISTORICO = 20        # peso (en días) de la frecuencia global en la historia "hasta ayer"
CLASES = ["sin_riesgo", "sin_bicis", "sin_anclajes"]

PARAMS_MODELO = dict(max_iter=150, learning_rate=0.05, min_samples_leaf=500, early_stopping=False, random_state=SEMILLA)



## Carga de datos

Se retoman los datos guardados de la libreta anterior.

In [16]:
def leer_datos():
    parquet_file = os.path.join(DATA_DIR, "p2_datos_cambios_de_capacidad.parquet")
    return pd.read_parquet(parquet_file)

In [17]:
cambios = leer_datos()

## Separación en entrenamiento, validación y test

Se usa la partición temporal común del proyecto, definida en `config.py`: entrenamiento hasta septiembre de 2024, validación de octubre de 2024 a septiembre de 2025 y prueba de octubre de 2025 a septiembre de 2026. Los bloques de 12 meses hacen que validación y prueba cubran todas las temporadas del año. Se asigna por día de servicio, y el día previo a cada frontera queda como brecha y no se usa: la etiqueta y las ventanas de P2 nunca cruzan el día, así que esto basta para que ningún conjunto use viajes del siguiente.

In [18]:
def asignar_particion(dia):
    """Regresa train, valid, test o brecha de cada día de servicio según la partición común del proyecto.
    Se asigna por día de servicio para no partir un día entre dos conjuntos; el día previo a cada frontera queda en brecha."""
    frontera_valid, frontera_test = pd.Timestamp(FRONTERA_VALID), pd.Timestamp(FRONTERA_TEST)
    brecha = pd.Timedelta(hours=BRECHA_HORAS)
    conjunto = np.select([dia < frontera_valid - brecha, dia < frontera_valid, dia < frontera_test - brecha, dia < frontera_test],
                         ["train", "brecha", "valid", "brecha"], default="test")
    return pd.Series(conjunto, index=dia.index)


particion = asignar_particion(cambios.dia)
for conjunto in ["train", "valid", "test", "brecha"]:
    dias = cambios.dia[particion == conjunto]
    print(f"{conjunto}: {dias.min().date()} a {dias.max().date()}")

train: 2020-01-01 a 2024-09-29
valid: 2024-10-01 a 2025-09-29
test: 2025-10-01 a 2026-09-30
brecha: 2024-09-30 a 2025-09-30


Se agrega una medición del riesgo histórico de que una estación se quede sin bicicletas o sin anclajes. La idea es: para cada fila (estación, día, hora), ¿qué tan seguido esa estación tuvo alguno de estos riesgos en esa misma hora y tipo de día (entre semana o fin de semana) en los días anteriores? Se calcula como un porcentaje respecto al número de registros anteriores. Esto se usará como feature adicional en el modelo real. Además, el baseline va a predecir este valor para cada riesgo.

A esta medición, se le agrega un parámetro PESO_HISTORICO que se usa para asignarle un peso a esta medición histórica.

In [ ]:
class HistoriaHastaAyer(BaseEstimator):
    """Frecuencia de cada clase en los días previos de la misma estación x tipo de día x hora, suavizada hacia la
    frecuencia global del periodo de entrenamiento. No usa el día actual ni el futuro.
    fit(): aprende solo la frecuencia global.
    transform(): calcula la historia sobre toda la tabla que reciba (necesita estacion, es_fin_semana, hora_idx,
    dia y clase)"""
 
    def __init__(self, peso_historico=20):
        self.peso_historico = peso_historico
 
    def fit(self, X, y=None):
        self.frec_global_ = {"sin_bicis": float((X["clase"] == 1).mean()),
                             "sin_anclajes": float((X["clase"] == 2).mean())}
        return self
 
    def transform(self, X):
        check_is_fitted(self, "frec_global_")
        
        X = X.sort_values(["estacion", "es_fin_semana", "hora_idx", "dia"]) 
        X["es_sin_bicis"] = (X["clase"] == 1).astype("int8")
        X["es_sin_anclajes"] = (X["clase"] == 2).astype("int8")
        
        # Se agrupa por estación, tipo de día, y hora, y se obtiene un conteo histórico de cada agrupación
        g = X.groupby(["estacion", "es_fin_semana", "hora_idx"], sort=False)
        dias_previos = g.cumcount()
        
        # Para cada clase de riesgo: se calcula la suma de incidencias restando el día actual
        # Esta suma se divide entre el número de días previos, y se obtiene un porcentaje histórico
        # de cuántos días hacia atrás han presentado el riesgo
        # Esto es exactamente lo que va a predecir el modelo baseline, y también serán features del modelo
        # Se agrega una variable que se usa para asignar peso a este histórico
        f = self.frec_global_
        X["hist_sin_bicis"] = (g["es_sin_bicis"].cumsum() - X["es_sin_bicis"] + self.peso_historico * f["sin_bicis"]) / (dias_previos + self.peso_historico)
        X["hist_sin_anclajes"] = (g["es_sin_anclajes"].cumsum() - X["es_sin_anclajes"] + self.peso_historico * f["sin_anclajes"]) / (dias_previos + self.peso_historico)
        return X
 

historia = HistoriaHastaAyer(peso_historico=PESO_HISTORICO).fit(cambios[particion == "train"])    # la frecuencia global sale solo del periodo de entrenamiento
cambios = historia.transform(cambios)
frec_global = historia.frec_global_

Contando con esto, se procede a separar los datos en train, test y val.

In [20]:
# Separar conjuntos de entrenamiento, validación, prueba (los días en brecha se descartan)
def separar_fit_val_test(cambios):
    particion = asignar_particion(cambios.dia)
    return cambios[particion == "train"], cambios[particion == "valid"], cambios[particion == "test"]
 
 
fit, val, test = separar_fit_val_test(cambios)
y_val, y_test = val.clase.to_numpy(), test.clase.to_numpy()
print("filas fit / val / test:", len(fit), len(val), len(test))
print("prevalencia test:", {CLASES[k]: round(float((y_test == k).mean()), 4) for k in (1, 2)})

filas fit / val / test: 18902160 4785372 5124710
prevalencia test: {'sin_bicis': 0.0074, 'sin_anclajes': 0.0103}


## Entrenamiento: Modelo Baseline

Como se comentó anteriormente, el baseline es la probabilidad de que para una estación, tipo de día (fin de semana o no) y hora, ocurra una clasificación de riesgo a partir del histórico.

In [21]:
# Baseline
class BaselineHistorico(BaseEstimator, ClassifierMixin):
    """Probabilidad de cada clase = su frecuencia histórica 'hasta ayer'. sin_riesgo es lo que sobra."""
 
    def fit(self, X, y=None):
        self.classes_ = np.array([0, 1, 2])
        return self
 
    def predict_proba(self, X):
        p_bicis, p_anclajes = X["hist_sin_bicis"].to_numpy(), X["hist_sin_anclajes"].to_numpy()
        return np.column_stack([1 - p_bicis - p_anclajes, p_bicis, p_anclajes])
 
    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(axis=1)]
 
 
baseline = BaselineHistorico().fit(fit[FEATURES_CON_HISTORICO], fit.clase)

## Entrenamiento: HistGradientBoostingClassifier

Se escogió un HistGradientBoostingClassifier por su capacidad para entrenar con cantidades masivas de datos. Se entrena a continuación.

In [22]:
modelo = HistGradientBoostingClassifier(**PARAMS_MODELO).fit(fit[FEATURES_CON_HISTORICO], fit.clase)
assert list(modelo.classes_) == [0, 1, 2]
print("iteraciones del modelo:", modelo.n_iter_)

iteraciones del modelo: 150


## Evaluación

### Elección de métricas
Las clases de riesgo son raras: en test, solo el 0.74% de las ventanas son sin_bicis y el 1.03% sin_anclajes. Con este desbalance, la exactitud no es útil: un modelo que siempre predice sin_riesgo tendría cerca de 98% de exactitud sin detectar un solo riesgo. Por eso se usan las siguientes métricas:

- PR-AUC por clase de riesgo y para riesgo en general (sin_bicis o sin_anclajes). Es la métrica principal: mide qué tan bien el modelo ordena las ventanas en riesgo por encima de las que no lo están, sin depender de un umbral. A diferencia del ROC-AUC, no se ve afectada por el desbalance en la clase negativa. Su referencia es la prevalencia de cada clase, que es lo que obtendría un modelo al azar.
- Log-loss: evalúa la calidad de las probabilidades. Es relevante porque el baseline y el modelo se comparan como probabilidades, y porque los umbrales de decisión se eligen sobre ellas.
- Precisión, recall y F1 por clase de riesgo, con umbrales elegidos en validación maximizando el F1 de cada clase. Traducen el modelo a la decisión del equipo de rebalanceo: el recall indica qué proporción de las estaciones en riesgo se detecta (riesgos no atendidos), y la precisión qué proporción de las alertas es real (viajes de rebalanceo desperdiciados). El F1 balancea ambos.
- Matriz de confusión normalizada por fila: muestra el recall de cada clase sin que sin_riesgo domine, y permite ver si el modelo confunde sin_bicis con sin_anclajes.

Todas las métricas se calculan en test; validación solo se usa para elegir los umbrales.

In [ ]:
 # Evaluación en test 
def calcular_metricas(y, p):
    """PR-AUC por clase de riesgo y de riesgo en general, y log-loss (no dependen del umbral)."""
    return {"ap_sin_bicis": average_precision_score(y == 1, p[:, 1]),
            "ap_sin_anclajes": average_precision_score(y == 2, p[:, 2]),
            "ap_riesgo": average_precision_score(y > 0, 1 - p[:, 0]),
            "log_loss": log_loss(y, p, labels=[0, 1, 2])}
 

# Se escogen los umbrales de corte con el conjunto de validación
def elegir_umbral(y_binaria, p):
    """Umbral que maximiza el F1 de una clase (se elige con validación)."""
    precision, recall, umbrales = precision_recall_curve(y_binaria, p)
    f1 = 2 * precision * recall / np.maximum(precision + recall, 1e-12)
    return umbrales[np.argmax(f1[:-1])]            # el último punto de la curva no tiene umbral asociado
 
 
def decidir_clase(p, umbrales):
    """Clase predicha: el riesgo cuyo score relativo a su umbral sea mayor y al menos 1."""
    score_bicis, score_anclajes = p[:, 1] / umbrales[1], p[:, 2] / umbrales[2]      # score relativo al umbral de cada clase
    pred = np.zeros(len(p), dtype=int)
    pred[(score_bicis >= 1) & (score_bicis >= score_anclajes)] = 1
    pred[(score_anclajes >= 1) & (score_anclajes > score_bicis)] = 2
    return pred
 
 
def evaluar_modelo(modelo, val, test, features):
    """Evalúa un modelo ya entrenado. Los umbrales de decisión salen de validación; las métricas, de test.
    Devuelve las probabilidades de test, la clase predicha, los umbrales y las métricas de ese modelo."""
    y_val, y_test = val.clase.to_numpy(), test.clase.to_numpy()
    p_val, p_test = modelo.predict_proba(val[features]), modelo.predict_proba(test[features])
    umbrales = {k: elegir_umbral(y_val == k, p_val[:, k]) for k in (1, 2)}
    pred = decidir_clase(p_test, umbrales)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, pred, labels=[1, 2], zero_division=0)
    metricas = {**calcular_metricas(y_test, p_test),
                "precision_sin_bicis": precision[0], "recall_sin_bicis": recall[0], "f1_sin_bicis": f1[0],
                "precision_sin_anclajes": precision[1], "recall_sin_anclajes": recall[1], "f1_sin_anclajes": f1[1]}
    return {"proba_test": p_test, "pred": pred, "umbrales": umbrales, "metricas": metricas}


def mostrar_evaluacion(nombre, resultado, y_test):
    """Imprime la evaluación de un modelo: métricas, reporte por clase y matriz de confusión."""
    print(f"\n=== {nombre} ===")
    print("MÉTRICAS EN TEST (PR-AUC y log-loss no dependen del umbral; precisión/recall/F1 sí)")
    print(pd.Series(resultado["metricas"]).round(4).to_string())
    print("\numbrales (elegidos en validación):", {CLASES[k]: round(float(v), 3) for k, v in resultado["umbrales"].items()})
    print(classification_report(y_test, resultado["pred"], target_names=CLASES, digits=3, zero_division=0))
    print("matriz de confusión (filas = real, columnas = predicho):\n",
          confusion_matrix(y_test, resultado["pred"], labels=[0, 1, 2]))
 
 
resultado_baseline = evaluar_modelo(baseline, val, test, FEATURES_CON_HISTORICO)
mostrar_evaluacion("baseline", resultado_baseline, y_test)

resultado_modelo = evaluar_modelo(modelo, val, test, FEATURES_CON_HISTORICO)
mostrar_evaluacion("modelo", resultado_modelo, y_test)
 


=== baseline ===
MÉTRICAS EN TEST (PR-AUC y log-loss no dependen del umbral; precisión/recall/F1 sí)
ap_sin_bicis              0.2453
ap_sin_anclajes           0.3982
ap_riesgo                 0.3329
log_loss                  0.0598
precision_sin_bicis       0.1690
recall_sin_bicis          0.4257
f1_sin_bicis              0.2419
precision_sin_anclajes    0.3905
recall_sin_anclajes       0.4371
f1_sin_anclajes           0.4125

umbrales (elegidos en validación): {'sin_bicis': 0.137, 'sin_anclajes': 0.17}
              precision    recall  f1-score   support

  sin_riesgo      0.990     0.977     0.983   5034260
   sin_bicis      0.169     0.426     0.242     37724
sin_anclajes      0.390     0.437     0.412     52726

    accuracy                          0.968   5124710
   macro avg      0.516     0.613     0.546   5124710
weighted avg      0.978     0.968     0.972   5124710

matriz de confusión (filas = real, columnas = predicho):
 [[4919626   78798   35836]
 [  21521   16059     1

Con la partición común del proyecto (entrenamiento hasta septiembre de 2024, validación de octubre de 2024 a septiembre de 2025 y prueba de octubre de 2025 a septiembre de 2026), el modelo no supera con claridad al baseline histórico.

En las medidas de ranking, que no dependen del umbral, empatan. El PR-AUC de riesgo en general es 0.333 en ambos y el log_loss es 0.0598 en ambos. En sin_anclajes el modelo gana por poco (0.403 contra 0.398), y en sin_bicis queda por debajo del baseline (0.232 contra 0.245). Las clases de riesgo siguen siendo raras: 0.74% y 1.03% de las ventanas en test.

Con los umbrales elegidos en validación, el modelo sí mejora el F1: sin_bicis sube de 0.242 a 0.278 y sin_anclajes de 0.412 a 0.415. La ganancia en sin_bicis viene sobre todo de la precisión (0.203 contra 0.169): el modelo genera 65,773 falsas alertas de sin_bicis, contra 78,798 del baseline, y detecta un poco más de casos reales (16,734 contra 16,059). La matriz de confusión confirma que casi nunca confunde una clase de riesgo con la otra (146 y 75 casos); sus errores son entre riesgo y no riesgo. Detecta cerca de 44% de los casos de sin_bicis y 45% de los de sin_anclajes, con una precisión de 20% y 39%.

Se define una función para obtener una matriz de confusión gráfica. Se usará para el registro en MLflow.

In [24]:
def p2_grafica_matriz_confusion(matriz: np.ndarray, clases: list, titulo: str):
    """Matriz de confusión: color y % por fila (recall de cada clase real), con el conteo debajo.
    Normalizar por fila evita que sin_riesgo, que es ~98% de los casos, aplaste a las clases raras."""
    matriz = np.asarray(matriz)
    por_fila = matriz / matriz.sum(axis=1, keepdims=True)
    figura, ax = plt.subplots(figsize=(6.5, 5.5))
    imagen = ax.imshow(por_fila, cmap="Blues", vmin=0, vmax=1)
    for i in range(len(clases)):
        for j in range(len(clases)):
            color = "white" if por_fila[i, j] > 0.5 else "black"
            ax.text(j, i, f"{por_fila[i, j]:.1%}\n({matriz[i, j]:,})", ha="center", va="center", color=color)
    ax.set_xticks(range(len(clases)), labels=clases)
    ax.set_yticks(range(len(clases)), labels=clases)
    ax.set_xlabel("Clase predicha")
    ax.set_ylabel("Clase real")
    ax.set_title(titulo)
    figura.colorbar(imagen, ax=ax, label="% de la clase real")
    figura.tight_layout()
    return figura

## Registro en MLflow

In [ ]:
import subprocess

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

from tc5061_proyecto_equipo07 import config as cfg
from tc5061_proyecto_equipo07 import dataset

# La ejecución de estas libretas registra en localhost
MLFLOW_TRACKING_URI = "http://localhost:5000"
MLFLOW_EXPERIMENTO = "p2_riesgo_estaciones"

# Parámetros del experimento que comparten todos los modelos
PARAMS_COMUNES = {"rango_datos": f"{cfg.MES_INICIO} a {cfg.MES_FIN}", "data_version": dataset.version_datos(),
                  "semilla": SEMILLA, "bin_min": BIN_MIN, "h_bins": H_BINS, "alfa": ALFA,
                  "frontera_valid": FRONTERA_VALID, "frontera_test": FRONTERA_TEST, "brecha_horas": BRECHA_HORAS,
                  "peso_historico": PESO_HISTORICO,
                  "n_variables": len(FEATURES_CON_HISTORICO), "variables": ",".join(FEATURES_CON_HISTORICO),
                  "datos_desde": str(cambios.dia.min().date()), "datos_hasta": str(cambios.dia.max().date()),
                  "filas_fit": len(fit), "filas_val": len(val), "filas_test": len(test)}

# Etiquetas que comparten los runs de P2 (stage, model_family y descripción van por modelo)
COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip() or "desconocido"
ETIQUETAS_COMUNES = {"equipo": "07", "pregunta": "P2", "tarea": "clasificacion", "commit": COMMIT,
                     "autor": "Fernando Arreola Treviño", "origen": "notebook"}


def registrar_modelo_en_mlflow(nombre, modelo, resultado, params_modelo, etiquetas_modelo, X_ejemplo, y_test,
                               params_comunes=PARAMS_COMUNES):
    """Un run de MLflow para un modelo: parámetros, métricas de test, matriz de confusión (imagen y conteos) y el modelo.
    nombre: nombre del run ('p2-baseline-historico' o 'p2-hgb').
    resultado: lo que devuelve evaluar_modelo (trae predicciones, métricas y umbrales).
    params_modelo: dict con los parámetros propios de ese modelo.
    etiquetas_modelo: stage, model_family y descripción del run.
    X_ejemplo: unas filas de FEATURES_CON_HISTORICO, para guardar la firma (columnas y tipos) del modelo.
    y_test: clases reales de test, para construir la matriz de confusión."""
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment(MLFLOW_EXPERIMENTO)
    umbrales = {f"umbral_{CLASES[k]}": float(v) for k, v in resultado["umbrales"].items()}     # salen de validación
    matriz = confusion_matrix(y_test, resultado["pred"], labels=[0, 1, 2])
    figura = p2_grafica_matriz_confusion(matriz, CLASES, f"Matriz de confusión: {nombre} (test)")
    with mlflow.start_run(run_name=nombre) as run:
        mlflow.set_tags({**ETIQUETAS_COMUNES, **etiquetas_modelo})
        mlflow.log_params({**params_comunes, **params_modelo, **umbrales})
        mlflow.log_metrics({f"test_{k}": float(v) for k, v in resultado["metricas"].items()})
        mlflow.log_figure(figura, "p2/matriz_confusion.png")
        mlflow.log_dict({"clases": CLASES, "filas": "clase real", "columnas": "clase predicha", "matriz": matriz.tolist()},
                        "p2/matriz_confusion.json")
        # cloudpickle: el formato por defecto (skops) rechaza la clase propia del baseline
        mlflow.sklearn.log_model(modelo, name="modelo",
                                 signature=infer_signature(X_ejemplo, modelo.predict_proba(X_ejemplo)),
                                 input_example=X_ejemplo, pyfunc_predict_fn="predict_proba",
                                 serialization_format="cloudpickle")
    plt.close(figura)
    print(f"{nombre}: run {run.info.run_id} registrado en {MLFLOW_TRACKING_URI}")
    return run.info.run_id


# Se llama por separado, un modelo a la vez:
X_ejemplo = fit[FEATURES_CON_HISTORICO].head(5)
registrar_modelo_en_mlflow("p2-baseline-historico", baseline, resultado_baseline, {"modelo": "baseline"},
                           {"stage": "baseline", "model_family": "frecuencia_historica",
                            "mlflow.note.content": "Frecuencia histórica 'hasta ayer' de cada riesgo (referencia)."},
                           X_ejemplo, y_test)
registrar_modelo_en_mlflow("p2-hgb", modelo, resultado_modelo,
                           {"modelo": "hgb", **{f"modelo_{k}": v for k, v in PARAMS_MODELO.items()},
                            "modelo_n_iter": modelo.n_iter_},
                           {"stage": "candidato", "model_family": "hist_gradient_boosting",
                            "mlflow.note.content": "HistGradientBoostingClassifier con las features y la historia 'hasta ayer'."},
                           X_ejemplo, y_test)

2026/10/18 11:24:59 INFO mlflow.tracking.fluent: Experiment with name 'p2_riesgo_estaciones' does not exist. Creating a new experiment.
c:\Users\ferna\Documents\python\mlops\TC5061-Proyecto-Equipo07\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/18 11:25:00 WARNING mlflow.sklearn: Saving scikit-learn m

🏃 View run p2-baseline-historico at: http://localhost:5000/#/experiments/1/runs/4a79a41e38eb4ec084413fd0fa718835
🧪 View experiment at: http://localhost:5000/#/experiments/1
p2-baseline-historico: run 4a79a41e38eb4ec084413fd0fa718835 registrado en http://localhost:5000


c:\Users\ferna\Documents\python\mlops\TC5061-Proyecto-Equipo07\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/18 11:25:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechan

🏃 View run p2-hgb at: http://localhost:5000/#/experiments/1/runs/0abb948ade2e43e6917cb43f5bb671a0
🧪 View experiment at: http://localhost:5000/#/experiments/1
p2-hgb: run 0abb948ade2e43e6917cb43f5bb671a0 registrado en http://localhost:5000


'0abb948ade2e43e6917cb43f5bb671a0'

----------------------------------